# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MUKUL-TIWARI/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# Setup

Connect to the FlyRank internship warehouse using DuckDB and a Hugging Face token stored securely in Colab Secrets.

In [18]:
%pip -q install duckdb huggingface_hub

In [19]:
import os
import getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)

print("Hugging Face token loaded successfully.")

Hugging Face token loaded successfully.


In [20]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("Connected successfully!")

Connected successfully!


In [21]:
query = f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {TABLES["fact_daily"]}
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,start_date,end_date
0,78835655,2025-01-27,2026-06-30


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of analysis

One row represents the daily performance of one content item for one client on one report date.

### Time window

For this assignment, I use the March 2026 partition (`month = '2026-03'`) as the analysis window. This is a mid-panel month, which avoids using the final month (June 2026) reserved as a natural test window. The daily performance table covers report dates from 2025-01-27 to 2026-06-30, but all feature development in this notebook is performed on the March 2026 partition.

In [22]:
query = f"""
SELECT
    COUNT(*) AS rows_in_march,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {TABLES["fact_daily"]}
WHERE month = '2026-03'
"""

con.sql(query).df()


,rows_in_march,start_date,end_date
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Feature

The initial features are created from the daily performance table using information available before the decision point. Candidate features include:

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_engaged_sessions

These features are historical measurements and are knowable at prediction time.

---

### Label / Proxy

The objective is to identify or rank content that may require optimization based on future performance. Any field derived from future outcomes or used to define the target is treated as the label and is never used as a feature.

---

### Context

Context fields are used for grouping, joining, or interpretation:

- client_hash_id
- content_hash_id
- report_date

These fields identify records and are not used as model features.

---

### Excluded

Excluded fields include:

- Future or label-derived columns (to prevent target leakage)
- Fields unavailable at the decision time

These fields are excluded because they would leak future information into the model.

In [23]:
query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    ga4_engaged_sessions,
    ga4_data_available
FROM {TABLES["fact_daily"]}
WHERE month = '2026-03'
LIMIT 10
"""

con.sql(query).df()


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions,ga4_data_available
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,<NA>,<NA>,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,<NA>,<NA>,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,<NA>,<NA>,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,<NA>,<NA>,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,<NA>,<NA>,<NA>
5,2026-03-01,client_73cda7b4e4f265ea,content_36c36abc7650d7af,239,1,7.347280,<NA>,<NA>,<NA>
6,2026-03-01,client_73cda7b4e4f265ea,content_a7da352b73b02668,191,0,7.832461,<NA>,<NA>,<NA>
7,2026-03-01,client_73cda7b4e4f265ea,content_05434271b257bb68,55,0,3.272727,<NA>,<NA>,<NA>
8,2026-03-01,client_73cda7b4e4f265ea,content_d056587ff7faca0c,77,0,5.636364,<NA>,<NA>,<NA>
9,2026-03-01,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2,0,4.500000,<NA>,<NA>,<NA>


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [24]:
query = f"""
DESCRIBE SELECT * FROM {TABLES["fact_daily"]}
LIMIT 1
"""

con.sql(query).df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [25]:
# Query 1 - Verify grain
query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM {TABLES["fact_daily"]}
WHERE month = '2026-03'
GROUP BY
    client_hash_id,
    content_hash_id,
    report_date
HAVING COUNT(*) > 1
LIMIT 10
"""

print("Query 1 - Grain")
display(con.sql(query).df())


# Query 2 - Verify date range
query = f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {TABLES["fact_daily"]}
WHERE month='2026-03'
"""

print("Query 2 - Date Range")
display(con.sql(query).df())


# Query 3 - Verify GA4 availability
query = f"""
SELECT
    COUNT(*) AS available_rows
FROM {TABLES["fact_daily"]}
WHERE month='2026-03'
AND ga4_data_available IS TRUE
"""

print("Query 3 - GA4 Availability")
display(con.sql(query).df())

Query 1 - Grain


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,row_count


Query 2 - Date Range


,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


Query 3 - GA4 Availability


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,413966


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

- Analysis is restricted to the March 2026 partition.
- Only rows where `ga4_data_available IS TRUE` are used.
- Future or label-derived fields are excluded to prevent leakage.
- June 2026 is reserved as the natural test period.

In [26]:
query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_rows
FROM {TABLES["fact_daily"]}
WHERE month = '2026-03'
"""

con.sql(query).df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_rows,gsc_rows
0,9841378,413966,3611061


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.